In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import importlib
import src.features
import src.quality_checks
import src.pipeline

importlib.reload(src.features)
importlib.reload(src.quality_checks)
importlib.reload(src.pipeline)

from src.pipeline import run_pipeline
from src.db import run_query

outputs = run_pipeline(refresh_data=False)

Step 1/7 — Creating schemas...
Step 2/7 — Reusing existing raw parquet files...
Step 3/7 — Loading raw parquet files into DuckDB...
Step 4/7 — Validating raw data project scope...
Running raw project-scope gate...
Raw project-scope checks:
- historical_required_columns: PASS
- forecast_required_columns: PASS
- historical_date_parse: PASS
- forecast_date_parse: PASS
- historical_city_scope: PASS
- forecast_city_scope: PASS
- historical_date_range: PASS
- forecast_date_range: PASS
- historical_rows_per_city: PASS
- forecast_rows_per_city: PASS
Raw project-scope gate passed.
Step 5/7 — Cleaning data, running quality checks, and building model features...
Running quality gate on cleaned historical data...
Quality checks:
- missing_values: PASS
- duplicate_rows: PASS
- duplicate_city_dates: PASS
- missing_dates: PASS
- weather_ranges: PASS
Quality gate passed.
Step 6/7 — Training models and building final 28-day forecast...
Training GradientBoosting model for ML horizon=1...
Training Gradie

In [2]:
run_query("""
SELECT table_schema, table_name
FROM information_schema.tables
ORDER BY table_schema, table_name
""")

,table_schema,table_name
0,analytics,final_28d_forecast
1,analytics,future_28d_forecast
2,analytics,model_features
3,raw,forecast
4,raw,historical


In [3]:
run_query("""
SELECT *
FROM analytics.final_28d_forecast
ORDER BY city, target_time
LIMIT 35
""")

,city,origin_time,forecast_horizon,target_time,source,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean
0,Baku,2026-04-29,1,2026-04-30,api_forecast,15.900000,0.000000,26.100000,70.000000,68.000000
1,Baku,2026-04-29,2,2026-05-01,api_forecast,16.700000,0.000000,32.400000,83.000000,59.000000
2,Baku,2026-04-29,3,2026-05-02,api_forecast,18.600000,0.000000,29.400000,84.000000,74.000000
3,Baku,2026-04-29,4,2026-05-03,api_forecast,13.900000,0.000000,28.200000,80.000000,82.000000
4,Baku,2026-04-29,5,2026-05-04,api_forecast,17.500000,0.000000,29.700000,85.000000,70.000000
5,Baku,2026-04-29,6,2026-05-05,api_forecast,19.700000,0.000000,25.400000,78.000000,68.000000
6,Baku,2026-04-29,7,2026-05-06,api_forecast,18.000000,0.000000,26.700000,75.000000,71.000000
7,Baku,2026-05-06,8,2026-05-07,ml_model,18.952488,1.006361,25.684269,74.555497,52.050364
8,Baku,2026-05-06,9,2026-05-08,ml_model,18.984226,0.968304,26.377488,76.677415,58.444950
9,Baku,2026-05-06,10,2026-05-09,ml_model,18.618464,1.605394,25.038844,75.178117,56.947301


In [4]:
run_query("""
SELECT city, source, COUNT(*) AS rows
FROM analytics.final_28d_forecast
GROUP BY city, source
ORDER BY city, source
""")

,city,source,rows
0,Baku,api_forecast,7
1,Baku,ml_model,21
2,Gabala,api_forecast,7
3,Gabala,ml_model,21
4,Guba,api_forecast,7
5,Guba,ml_model,21
6,Lankaran,api_forecast,7
7,Lankaran,ml_model,21
8,Shaki,api_forecast,7
9,Shaki,ml_model,21
